In [26]:
from os import getcwd, listdir
from os.path import join, dirname, exists, isdir
import pandas as pd
from scipy.stats import wasserstein_distance
import numpy as np
from scipy.spatial.distance import cdist
import ot
from sklearn.cluster import KMeans
from sklearn.neighbors import KernelDensity
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.dates import date2num, DateFormatter, MonthLocator
import json
import pickle
from collections import defaultdict
from itertools import product, cycle
from pprint import pprint
from math import ceil, sqrt
import textwrap
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import random
from typing import List, Tuple, Optional, Dict, Any
import re
# from matplotlib import colormaps

In [2]:
# Define all relevant electricity use columns for the table
all_electricity_cols = [
    "End Use: Electricity: Heating", "End Use: Electricity: Heating Fans/Pumps",
    "End Use: Electricity: Heating Heat Pump Backup", "End Use: Electricity: Heating Heat Pump Backup Fans/Pumps",
    "End Use: Electricity: Cooling", "End Use: Electricity: Cooling Fans/Pumps",
    "End Use: Electricity: Hot Water", "End Use: Electricity: Lighting Interior",
    "End Use: Electricity: Lighting Exterior", "End Use: Electricity: Appliances",
    "End Use: Electricity: Plug Loads", "End Use: Electricity: Photovoltaics"
]

def find_peak_electricity_use(region_name, num_residences, residence_type="all", casename=None, offset_needed=False):
    """
    Searches through RESSTOCK building data to find the single building 
    with the highest total electricity consumption at a single hour.

    Args:
        region_name (str): The name of the city for the data.
        num_residences (int): The total number of residences in the dataset.
        residence_type (str, optional): The type of residence. Defaults to "all".
        casename (str, optional): The case name for the dataset. Defaults to None.
        offset_needed (bool, optional): Indicates if an offset is needed for file paths. Defaults to False.

    Returns:
        tuple: A tuple containing a string with a summary of the peak-use building 
               and a pandas DataFrame of the electricity use breakdown for that hour.
    """
    
    max_total_electricity = 0
    peak_hour_data = None
    peak_building_id = None
    peak_building_metadata = None

    
                
    # Retrieve additional metadata from the centralized buildstock CSV file
    resstock_buildstock_data_path = join("resources", "national", "all_national_8000_all_unit_buildstock.csv")
    resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)
    
    residences_to_collect = range(num_residences)

    for i in residences_to_collect:
        if offset_needed:
            j = i + 1 + num_residences
        else:
            j = i + 1
        
        filepath = join(f"{region_name}_{num_residences}_{residence_type}_unit_residence{'' if casename is None else f'_{casename}'}", f'run{j}','run', 'results_timeseries.csv')
        
        try:
            timeseries_data = pd.read_csv(filepath, low_memory=True, header=[0,1])
        except Exception as e:
            continue

        # Clean up column names by dropping the second level
        timeseries_data.columns = timeseries_data.columns.droplevel(1)

        # Check for the existence of the total electricity column
        if "Fuel Use: Electricity: Total" in timeseries_data.columns:
            # Find the index of the maximum total electricity consumption
            max_index = timeseries_data["Fuel Use: Electricity: Total"].idxmax()
            current_max_use = timeseries_data.loc[max_index]["Fuel Use: Electricity: Total"]
            
            # Compare and update if a new peak is found
            if current_max_use > max_total_electricity:
                max_total_electricity = current_max_use
                peak_hour_data = timeseries_data.loc[max_index]
                
                # Get the building ID from the JSON metadata file
                data_point_out_filepath = join(f"{region_name}_{num_residences}_{residence_type}_unit_residence{'' if casename is None else f'_{casename}'}", f'run{j}','run', 'data_point_out.json')
                try:
                    with open(data_point_out_filepath, "r") as f:
                        json_data = json.load(f)
                        peak_building_id = json_data["BuildExistingModel"]["building_id"]
                except Exception as e:
                    peak_building_id = "N/A"
                peak_building_metadata = resstock_buildstock_data[resstock_buildstock_data["Building"] == peak_building_id]
                
    # Prepare the output
    if peak_hour_data is not None:        
        return peak_hour_data, peak_building_metadata
    else:
        return "No data found to process.", None

In [21]:
peak_hour_data, peak_building_metadata  = find_peak_electricity_use("national_ASHP", 8000, casename=None)

/tmp/ipykernel_2178/1957883278.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [22]:
# Define all relevant electricity use columns for the table
all_electricity_cols = [
    "End Use: Electricity: Heating", "End Use: Electricity: Heating Fans/Pumps",
    "End Use: Electricity: Heating Heat Pump Backup", "End Use: Electricity: Heating Heat Pump Backup Fans/Pumps",
    "End Use: Electricity: Cooling", "End Use: Electricity: Cooling Fans/Pumps",
    "End Use: Electricity: Hot Water", "End Use: Electricity: Lighting Interior",
    "End Use: Electricity: Lighting Exterior", "End Use: Electricity: Appliances",
    "End Use: Electricity: Plug Loads", "End Use: Electricity: Photovoltaics"
]

# Create the breakdown table for the peak hour
breakdown_data = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data.index],
    "Electricity (kWh)": [peak_hour_data[col] for col in all_electricity_cols if col in peak_hour_data.index]
}
breakdown_table = pd.DataFrame(breakdown_data)

breakdown_table

# # Generate a summary string
# peak_time = pd.to_datetime(peak_hour_data["Time"])
# summary = f"🥇 The building with the highest peak electricity consumption is building ID: {peak_building_metadata['Building']}\n"
# summary += f"Location: {peak_building_metadata['city'].iloc[0]}, {peak_building_metadata['state'].iloc[0]}\n"
# summary += f"The peak consumption was {peak_hour_data['Fuel Use: Electricity: Total']:.2f} kWh at {peak_time.strftime('%Y-%m-%d %H:%M')}.\n\n"
# summary += f"Breakdown of Electricity Use during Peak Hour:\n"

# summary

,End Use,Electricity (kWh)
0,Heating,18.751
1,Heating Fans/Pumps,1.858
2,Heating Heat Pump Backup,81.240
3,Heating Heat Pump Backup Fans/Pumps,1.964
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Hot Water,0.000


In [23]:
peak_hour_data_ASHP, peak_building_metadata_ASHP  = find_peak_electricity_use("national_ASHP", 8000, casename="ASHP")

/tmp/ipykernel_2178/1957883278.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [24]:
# Create the breakdown table for the peak hour
breakdown_data_ASHP = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data_ASHP.index],
    "Electricity (kWh)": [peak_hour_data_ASHP[col] for col in all_electricity_cols if col in peak_hour_data_ASHP.index]
}
breakdown_table_ASHP = pd.DataFrame(breakdown_data_ASHP)

breakdown_table_ASHP

,End Use,Electricity (kWh)
0,Heating,19.106
1,Heating Fans/Pumps,1.845
2,Heating Heat Pump Backup,82.193
3,Heating Heat Pump Backup Fans/Pumps,1.978
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Hot Water,0.000


In [33]:
peak_hour_data_ASHPautosizing, peak_building_metadata_ASHPautosizing  = find_peak_electricity_use("national_ASHP", 8000, casename="ASHPautosizing")

/tmp/ipykernel_2178/1957883278.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [34]:
# Create the breakdown table for the peak hour
breakdown_data_ASHPautosizing = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data_ASHPautosizing.index],
    "Electricity (kWh)": [peak_hour_data_ASHPautosizing[col] for col in all_electricity_cols if col in peak_hour_data_ASHPautosizing.index]
}
breakdown_table_ASHPautosizing = pd.DataFrame(breakdown_data_ASHPautosizing)

breakdown_table_ASHPautosizing

,End Use,Electricity (kWh)
0,Heating,19.106
1,Heating Fans/Pumps,1.845
2,Heating Heat Pump Backup,82.193
3,Heating Heat Pump Backup Fans/Pumps,1.978
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Hot Water,0.000


In [25]:
peak_building_metadata

,Building,ASHRAE IECC Climate Zone 2004,Bathroom Spot Vent Hour,County and PUMA,AIANNH Area,CEC Climate Zone,City,County,AHS Region,ASHRAE IECC Climate Zone 2004 - 2A Split,...,Misc Pool Pump,Refrigerator,Roof Material,State Metro Median Income,Water Heater Fuel,Water Heater Efficiency,Water Heater In Unit,Water Heater Location,Window Areas,Windows
3494,3495,3A,Hour19,"G3701250, G37003700",No,NaN,In another census Place,"NC, Moore County",Non-CBSA South Atlantic,3A,...,NaN,EF 19.9,"Asphalt Shingles, Medium",Not Available,Electricity,Electric Standard,Yes,Living Space,F12 B12 L12 R12,"Single, Clear, Non-metal"


In [26]:
peak_building_metadata_ASHP

,Building,ASHRAE IECC Climate Zone 2004,Bathroom Spot Vent Hour,County and PUMA,AIANNH Area,CEC Climate Zone,City,County,AHS Region,ASHRAE IECC Climate Zone 2004 - 2A Split,...,Misc Pool Pump,Refrigerator,Roof Material,State Metro Median Income,Water Heater Fuel,Water Heater Efficiency,Water Heater In Unit,Water Heater Location,Window Areas,Windows
3494,3495,3A,Hour19,"G3701250, G37003700",No,NaN,In another census Place,"NC, Moore County",Non-CBSA South Atlantic,3A,...,NaN,EF 19.9,"Asphalt Shingles, Medium",Not Available,Electricity,Electric Standard,Yes,Living Space,F12 B12 L12 R12,"Single, Clear, Non-metal"


In [35]:
peak_building_metadata_ASHPautosizing

,Building,ASHRAE IECC Climate Zone 2004,Bathroom Spot Vent Hour,County and PUMA,AIANNH Area,CEC Climate Zone,City,County,AHS Region,ASHRAE IECC Climate Zone 2004 - 2A Split,...,Misc Pool Pump,Refrigerator,Roof Material,State Metro Median Income,Water Heater Fuel,Water Heater Efficiency,Water Heater In Unit,Water Heater Location,Window Areas,Windows
3494,3495,3A,Hour19,"G3701250, G37003700",No,NaN,In another census Place,"NC, Moore County",Non-CBSA South Atlantic,3A,...,NaN,EF 19.9,"Asphalt Shingles, Medium",Not Available,Electricity,Electric Standard,Yes,Living Space,F12 B12 L12 R12,"Single, Clear, Non-metal"


In [27]:
peak_hour_data_ASHP

Time                                                                                    2022-02-04T21:00:00
TimeDST                                                                                 2022-02-04T21:00:00
TimeUTC                                                                                2022-02-05T02:00:00Z
Energy Use: Total                                                                                    358.79
Energy Use: Net                                                                                      358.79
Fuel Use: Electricity: Total                                                                        105.151
Fuel Use: Electricity: Net                                                                          105.151
End Use: Electricity: Heating                                                                        19.106
End Use: Electricity: Heating Fans/Pumps                                                              1.845
End Use: Electricity: Heatin

In [36]:
peak_hour_data_ASHPautosizing

Time                                                                                    2022-02-04T21:00:00
TimeDST                                                                                 2022-02-04T21:00:00
TimeUTC                                                                                2022-02-05T02:00:00Z
Energy Use: Total                                                                                    358.79
Energy Use: Net                                                                                      358.79
Fuel Use: Electricity: Total                                                                        105.151
Fuel Use: Electricity: Net                                                                          105.151
End Use: Electricity: Heating                                                                        19.106
End Use: Electricity: Heating Fans/Pumps                                                              1.845
End Use: Electricity: Heatin

In [31]:
peak_building_metadata_ASHP['Geometry Floor Area']

3494    4000+
Name: Geometry Floor Area, dtype: object

In [3]:
peak_hour_data_ASHP_8_autosizing, peak_building_metadata_ASHP_8_autosizing  = find_peak_electricity_use("national_ASHP", 8, casename="ASHPautosizing")

/tmp/ipykernel_1084/685573527.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [4]:
# Create the breakdown table for the peak hour
breakdown_data_ASHP_8_autosizing = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data_ASHP_8_autosizing.index],
    "Electricity (kWh)": [peak_hour_data_ASHP_8_autosizing[col] for col in all_electricity_cols if col in peak_hour_data_ASHP_8_autosizing.index]
}
breakdown_table_ASHP_8_autosizing = pd.DataFrame(breakdown_data_ASHP_8_autosizing)

breakdown_table_ASHP_8_autosizing

,End Use,Electricity (kWh)
0,Heating,1.103
1,Heating Fans/Pumps,0.026
2,Heating Heat Pump Backup,11.838
3,Heating Heat Pump Backup Fans/Pumps,0.162
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Lighting Interior,0.145
7,Lighting Exterior,0.005
8,Plug Loads,0.521


In [5]:
peak_hour_data_ASHP_8_autosizingchangingyml, peak_building_metadata_ASHP_8_autosizingchangingyml  = find_peak_electricity_use("national_ASHP", 8, casename="ASHPautosizingchangingyml")

/tmp/ipykernel_1084/685573527.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [6]:
# Create the breakdown table for the peak hour
breakdown_data_ASHP_8_autosizingchangingyml = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data_ASHP_8_autosizingchangingyml.index],
    "Electricity (kWh)": [peak_hour_data_ASHP_8_autosizingchangingyml[col] for col in all_electricity_cols if col in peak_hour_data_ASHP_8_autosizingchangingyml.index]
}
breakdown_table_ASHP_8_autosizingchangingyml = pd.DataFrame(breakdown_data_ASHP_8_autosizingchangingyml)

breakdown_table_ASHP_8_autosizingchangingyml

,End Use,Electricity (kWh)
0,Heating,1.103
1,Heating Fans/Pumps,0.026
2,Heating Heat Pump Backup,11.838
3,Heating Heat Pump Backup Fans/Pumps,0.162
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Lighting Interior,0.145
7,Lighting Exterior,0.005
8,Plug Loads,0.521


In [17]:
def extract_capacity_data(base_folder: str) -> Tuple[List[float], List[float]]:
    """
    Traverses the base folder, finds run-numbered subdirectories (run1, run2, etc.),
    and extracts heating and cooling capacity data from 'data_point_out.json'.

    Args:
        base_folder: The root directory where run folders are located.

    Returns:
        A tuple containing two lists: (heating_capacities, cooling_capacities).
    """
    print(f"Starting data extraction from: {base_folder}")
    
    heating_capacities = []
    cooling_capacities = []
    
    # Regex to match 'run1', 'run23', etc.
    run_folder_pattern = re.compile(r"run\d+$")

    for item_name in listdir(base_folder):
        # Construct the full path to the potential run folder
        run_folder_path = join(base_folder, item_name)

        # 1. Check if the item is a directory and matches the 'run#' pattern
        if isdir(run_folder_path) and run_folder_pattern.match(item_name):
            
            # 2. Construct the path to the JSON output file
            json_file_path = join(run_folder_path, "run", "data_point_out.json")

            if not exists(json_file_path):
                print(f"  [MISSING] {item_name}: data_point_out.json not found.")
                continue

            try:
                # 3. Read and parse the JSON file
                with open(json_file_path, 'r') as f:
                    data = json.load(f)

                # 4. Extract the nested capacity values
                # Path: data -> "UpgradeCosts" -> "size_..."
                upgrade_costs = data.get("UpgradeCosts", {})
                
                heating_capacity = upgrade_costs.get("size_heating_system_primary_k_btu_h")
                cooling_capacity = upgrade_costs.get("size_cooling_system_primary_k_btu_h")
                
                if heating_capacity is not None and cooling_capacity is not None:
                    # Append successfully extracted numeric data
                    heating_capacities.append(float(heating_capacity))
                    cooling_capacities.append(float(cooling_capacity))
                    # print(f"  [SUCCESS] {item_name}: H={heating_capacity:.2f}, C={cooling_capacity:.2f}")
                else:
                    print(f"  [WARN] {item_name}: Capacity data missing or incomplete in 'UpgradeCosts' object.")

            except json.JSONDecodeError:
                print(f"  [ERROR] {item_name}: Failed to decode JSON file.")
            except FileNotFoundError:
                # Should be caught by the exists check, but good practice
                print(f"  [ERROR] {item_name}: JSON file not found at {json_file_path}.")
            except Exception as e:
                print(f"  [ERROR] {item_name}: An unexpected error occurred: {e}")

    print(f"\nExtraction complete. Found {len(heating_capacities)} valid data points.")
    return heating_capacities, cooling_capacities

def create_histograms(heating_data: List[float], cooling_data: List[float], output_dir: str):
    """
    Generates and saves histograms for the extracted capacity data.
    
    Args:
        heating_data: List of heating capacity values.
        cooling_data: List of cooling capacity values.
        output_dir: Directory where the plots will be saved.
    """
    # Convert to NumPy arrays for easier plotting
    heating_data_np = np.array(heating_data)
    cooling_data_np = np.array(cooling_data)

    if not heating_data_np.size or not cooling_data_np.size:
        print("No valid data points found to create histograms.")
        return

    # --- Plotting Setup ---
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # --- Heating Capacity Histogram ---
    plt.figure(figsize=(10, 6))
    
    # Determine bin size based on range or fixed number
    bins = min(50, len(heating_data) // 5) # Use max 50 bins or fewer if data is sparse
    
    plt.hist(heating_data_np, bins=bins, color='#1f77b4', edgecolor='black', alpha=0.7)
    plt.title('Distribution of Primary Heating System Capacity (kBtu/h)', fontsize=16)
    plt.xlabel('Heating Capacity (kBtu/h)', fontsize=14)
    plt.ylabel('Frequency (Count of Runs)', fontsize=14)
    plt.axvline(np.mean(heating_data_np), color='r', linestyle='dashed', linewidth=2, label=f'Mean: {np.mean(heating_data_np):.2f} kBtu/h')
    plt.legend()
    plt.grid(axis='y', alpha=0.5)
    
    heating_plot_path = join(output_dir, "heating_capacity_histogram.png")
    plt.savefig(heating_plot_path, bbox_inches='tight')
    plt.close()
    print(f"Saved Heating Capacity Histogram to: {heating_plot_path}")

    # --- Cooling Capacity Histogram ---
    plt.figure(figsize=(10, 6))
    
    # Determine bin size based on range or fixed number
    bins = min(50, len(cooling_data) // 5) # Use max 50 bins or fewer if data is sparse

    plt.hist(cooling_data_np, bins=bins, color='#ff7f0e', edgecolor='black', alpha=0.7)
    plt.title('Distribution of Primary Cooling System Capacity (kBtu/h)', fontsize=16)
    plt.xlabel('Cooling Capacity (kBtu/h)', fontsize=14)
    plt.ylabel('Frequency (Count of Runs)', fontsize=14)
    plt.axvline(np.mean(cooling_data_np), color='r', linestyle='dashed', linewidth=2, label=f'Mean: {np.mean(cooling_data_np):.2f} kBtu/h')
    plt.legend()
    plt.grid(axis='y', alpha=0.5)
    
    cooling_plot_path = join(output_dir, "cooling_capacity_histogram.png")
    plt.savefig(cooling_plot_path, bbox_inches='tight')
    plt.close()
    print(f"Saved Cooling Capacity Histogram to: {cooling_plot_path}")

In [21]:
# base_dir = join(getcwd(), "national_ASHP_8_all_unit_residence_ASHPautosizing")
# base_dir = join(getcwd(), "national_ASHP_8_all_unit_residence_ASHPautosizingchangingyml")
# base_dir = join(getcwd(), "national_ASHP_8000_all_unit_residence_ASHP")
base_dir = join(getcwd(), "national_ASHP_8000_all_unit_residence_ASHPautosizing")

heating_data, cooling_data = extract_capacity_data(base_dir)
create_histograms(heating_data, cooling_data, base_dir)

Starting data extraction from: /home/arnavgautam/resstock-3.4.0/national_ASHP_8000_all_unit_residence_ASHPautosizing
  [WARN] run2039: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7507: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run6483: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7809: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run4468: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7071: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run4738: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run1494: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run742: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run232: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run5885: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WA

In [31]:
FLOOR_AREA_MIDPOINTS = {
    "0-499": 250,
    "500-749": 625,
    "750-999": 875,
    "1000-1499": 1250,
    "1500-1999": 1750,
    "2000-2499": 2250,
    "2500-2999": 2750,
    "3000-3999": 3500,
    "4000+": 4000,
}

def extract_capacity_data_as_list_of_dicts(base_folder: str) -> List[Dict[str, Any]]:
    """
    Traverses the base folder, finds run-numbered subdirectories (run1, run2, etc.),
    and extracts heating and cooling capacity data from 'data_point_out.json'.
    
    Now returns a list of dictionaries, including the 'run_id' (Building number)
    for merging with the CSV file.

    Args:
        base_folder: The root directory where run folders are located.

    Returns:
        A list of dictionaries, each containing 'run_id', 'heating_capacity', 
        and 'cooling_capacity'.
    """
    print(f"Starting data extraction from: {base_folder}")
    
    capacity_data_list = []
    
    # Regex to match 'run1', 'run23', etc.
    run_folder_pattern = re.compile(r"run\d+$")

    for item_name in listdir(base_folder):
        run_folder_path = join(base_folder, item_name)

        # 1. Check if the item is a directory and matches the 'run#' pattern
        if isdir(run_folder_path) and run_folder_pattern.match(item_name):
            
            # Extract the run number (Building ID) from the folder name
            match = re.search(r'\d+', item_name)
            if not match:
                continue
            run_number = int(match.group(0))

            # 2. Construct the path to the JSON output file
            json_file_path = join(run_folder_path, "run", "data_point_out.json")

            if not exists(json_file_path):
                print(f"  [MISSING] {item_name}: data_point_out.json not found.")
                continue

            try:
                # 3. Read and parse the JSON file
                with open(json_file_path, 'r') as f:
                    data = json.load(f)

                # 4. Extract the nested capacity values
                upgrade_costs = data.get("UpgradeCosts", {})
                
                heating_capacity = upgrade_costs.get("size_heating_system_primary_k_btu_h")
                cooling_capacity = upgrade_costs.get("size_cooling_system_primary_k_btu_h")
                
                if heating_capacity is not None and cooling_capacity is not None:
                    # Append successfully extracted numeric data along with the run_id
                    capacity_data_list.append({
                        "Building": run_number, # Renamed key to match CSV column name
                        "heating_capacity": float(heating_capacity),
                        "cooling_capacity": float(cooling_capacity)
                    })
                else:
                    print(f"  [WARN] {item_name}: Capacity data missing or incomplete in 'UpgradeCosts' object.")

            except json.JSONDecodeError:
                print(f"  [ERROR] {item_name}: Failed to decode JSON file.")
            except FileNotFoundError:
                print(f"  [ERROR] {item_name}: JSON file not found at {json_file_path}.")
            except Exception as e:
                print(f"  [ERROR] {item_name}: An unexpected error occurred: {e}")

    print(f"\nExtraction complete. Found {len(capacity_data_list)} valid data points.")
    return capacity_data_list

def create_boxplot_by_floor_area(capacity_data: List[dict], csv_path: str, floor_area_map: dict, output_dir: str):
    """
    Generates box and whisker plots of heating/cooling capacity categorized by floor area.
    
    Args:
        capacity_data: List of dictionaries containing 'Building', 'heating_capacity', 
                       and 'cooling_capacity'.
        csv_path: Path to the main results CSV file.
        floor_area_map: Dictionary defining floor area categories and their order.
        output_dir: Directory where the plots will be saved.
    """
    print(f"\nStarting analysis for floor area relationship using CSV: {csv_path}")

    if not exists(csv_path):
        print(f"  [ERROR] CSV file not found at: {csv_path}. Skipping box plots.")
        return

    # Convert capacity data to DataFrame for merging
    capacity_df = pd.DataFrame(capacity_data)

    try:
        # Load the results CSV, only need the 'Building' and floor area columns
        results_df = pd.read_csv(csv_path, usecols=['Building', 'Geometry Floor Area'], low_memory=False)
        
        # Merge the capacity data with the floor area data using 'Building' as the key
        merged_df = pd.merge(capacity_df, results_df, on='Building', how='inner')
        
    except Exception as e:
        print(f"  [ERROR] Failed to load or merge dataframes: {e}")
        return

    if merged_df.empty:
        print("  [WARN] No matching data found after merging capacity data and CSV. Check 'Building' IDs and column names.")
        return

    # Filter data to only include categories defined in the map
    category_order = list(floor_area_map.keys())
    merged_df = merged_df[merged_df['Geometry Floor Area'].isin(category_order)].copy()

    if merged_df.empty:
        print("  [WARN] No data points matched the specified floor area categories. Skipping box plots.")
        return

    # Set the floor area column as a categorical type with the defined order for correct plotting
    merged_df['Geometry Floor Area'] = pd.Categorical(
        merged_df['Geometry Floor Area'], 
        categories=category_order, 
        ordered=True
    )
    
    # Sort the dataframe by the new categorical column for plot order
    merged_df.sort_values('Geometry Floor Area', inplace=True)

    # --- Plotting Function ---
    def plot_capacity_boxplot(data_frame, capacity_col, title, filename, color):
        plt.figure(figsize=(12, 7))
        
        # Use pandas boxplot function on the sorted categorical data
        data_frame.boxplot(
            column=[capacity_col], 
            by='Geometry Floor Area', 
            ax=plt.gca(), 
            grid=False,
            vert=True,
            patch_artist=True,
            showfliers=False, # Hide outliers for clarity
            meanline=True, 
            showmeans=True,
            medianprops=dict(color='black', linewidth=1.5),
            meanprops=dict(color='red', linestyle='--', linewidth=1.5),
            boxprops=dict(facecolor=color, alpha=0.7),
            rot=45 # Rotate x-labels
        )

        # Clean up Matplotlib figure details
        plt.suptitle('') # Remove automatic pandas suptitle
        plt.title(title, fontsize=16)
        plt.xlabel('Floor Area Category (sq ft)', fontsize=14)
        plt.ylabel('Capacity (kBtu/h)', fontsize=14)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        plot_path = join(output_dir, filename)
        plt.savefig(plot_path)
        plt.close()
        print(f"Saved {title} Box Plot to: {plot_path}")

    # 1. Heating Capacity Box Plot
    plot_capacity_boxplot(
        merged_df, 
        'heating_capacity', 
        'Heating System Capacity Distribution by Floor Area', 
        "heating_capacity_boxplot_by_floor_area.png",
        '#1f77b4' # Blue
    )

    # 2. Cooling Capacity Box Plot
    plot_capacity_boxplot(
        merged_df, 
        'cooling_capacity', 
        'Cooling System Capacity Distribution by Floor Area', 
        "cooling_capacity_boxplot_by_floor_area.png",
        '#ff7f0e' # Orange
    )

In [32]:
base_dir = join(getcwd(), "national_ASHP_8000_all_unit_residence_ASHP")

capacity_data = extract_capacity_data_as_list_of_dicts(base_dir)
capacity_data

Starting data extraction from: /home/arnavgautam/resstock-3.4.0/national_ASHP_8000_all_unit_residence_ASHP
  [WARN] run2039: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7507: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run6483: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7809: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run4468: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run7071: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run4738: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run1494: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run742: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run232: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run5885: Capacity data missing or incomplete in 'UpgradeCosts' object.
  [WARN] run214

[{'Building': 3667, 'heating_capacity': 23.85, 'cooling_capacity': 23.85},
 {'Building': 6866, 'heating_capacity': 55.49, 'cooling_capacity': 55.49},
 {'Building': 2142, 'heating_capacity': 28.9, 'cooling_capacity': 28.9},
 {'Building': 6605, 'heating_capacity': 28.94, 'cooling_capacity': 28.94},
 {'Building': 2147, 'heating_capacity': 33.31, 'cooling_capacity': 33.31},
 {'Building': 2817, 'heating_capacity': 5.8, 'cooling_capacity': 5.8},
 {'Building': 3791, 'heating_capacity': 71.48, 'cooling_capacity': 71.48},
 {'Building': 2984, 'heating_capacity': 36.27, 'cooling_capacity': 36.27},
 {'Building': 2566, 'heating_capacity': 70.24, 'cooling_capacity': 70.24},
 {'Building': 437, 'heating_capacity': 57.89, 'cooling_capacity': 57.89},
 {'Building': 4569, 'heating_capacity': 27.05, 'cooling_capacity': 27.05},
 {'Building': 1866, 'heating_capacity': 45.2, 'cooling_capacity': 45.2},
 {'Building': 6427, 'heating_capacity': 27.75, 'cooling_capacity': 27.75},
 {'Building': 640, 'heating_capaci

In [33]:
create_boxplot_by_floor_area(capacity_data, join(getcwd(), "resources", "national", "all_national_8000_all_unit_buildstock.csv"), FLOOR_AREA_MIDPOINTS, base_dir)


Starting analysis for floor area relationship using CSV: /home/arnavgautam/resstock-3.4.0/resources/national/all_national_8000_all_unit_buildstock.csv
Saved Heating System Capacity Distribution by Floor Area Box Plot to: /home/arnavgautam/resstock-3.4.0/national_ASHP_8000_all_unit_residence_ASHP/heating_capacity_boxplot_by_floor_area.png
Saved Cooling System Capacity Distribution by Floor Area Box Plot to: /home/arnavgautam/resstock-3.4.0/national_ASHP_8000_all_unit_residence_ASHP/cooling_capacity_boxplot_by_floor_area.png
